# NFL Game Prediction — Modeling

This notebook trains and evaluates machine learning models for predicting the probability that the home team wins an NFL game.

The models use pre-game features created in the previous notebook:

- `win_pct_diff`
- `ppg_diff`
- `defense_diff`

The data is split chronologically to prevent future games from influencing predictions about earlier games.

The initial models are:

1. Logistic Regression
2. Random Forest
3. Gradient Boosting

The primary evaluation metrics are **Log Loss** and **Brier Score**, since the goal of this project is to produce reliable win probabilities rather than only predicting the winning team.

## 1. Imports

In [48]:
import polars as pl

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss

## 2. Load the Feature-Engineered Dataset

The feature engineering notebook saved the processed dataset as a Parquet file.

This dataset contains only information that was available before each game.

In [49]:
model_data = pl.read_parquet(
    "../data/processed/model_data.parquet"
)

print(model_data.shape)

(2718, 68)


In [50]:
model_data.head()

game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,home_score,location,result,total,overtime,old_game_id,gsis,nfl_detail_id,pfr,pff,espn,ftn,away_rest,home_rest,away_moneyline,home_moneyline,spread_line,away_spread_odds,home_spread_odds,total_line,under_odds,over_odds,div_game,roof,surface,temp,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium,home_win,home_win_pct,home_ppg,home_papg,away_win_pct,away_ppg,away_papg,win_pct_diff,ppg_diff,defense_diff,home_rolling_win_pct_5,home_rolling_ppg_5,home_rolling_papg_5,home_rolling_point_diff_5,away_rolling_win_pct_5,away_rolling_ppg_5,away_rolling_papg_5,away_rolling_point_diff_5,rolling_win_pct_diff_5,rolling_ppg_diff_5,rolling_defense_diff_5,rolling_point_diff_diff_5
str,i32,str,i32,str,str,str,str,i32,str,i32,str,i32,i32,i32,str,i32,str,str,i32,str,i32,i32,i32,i32,i32,f64,i32,i32,f64,i32,i32,i32,str,str,i32,i32,str,str,str,str,str,str,str,str,str,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2015_02_DEN_KC""",2015,"""REG""",2,"""2015-09-17""","""Thursday""","""20:25""","""DEN""",31,"""KC""",24,"""Home""",-7,55,0,"""2015091700""",56519,null,"""201509170kan""",3422,"""400791624""",null,4,4,146,-162,3.0,102,-113,42.0,100,-110,1,"""outdoors""","""grass""",87,13,"""00-0010346""","""00-0023436""","""Peyton Manning""","""Alex Smith""","""Gary Kubiak""","""Andy Reid""","""Walt Anderson""","""KAN00""","""Arrowhead Stadium""",0,1.0,27.0,20.0,1.0,19.0,13.0,0.0,8.0,-7.0,1.0,27.0,20.0,7.0,1.0,19.0,13.0,6.0,0.0,8.0,-7.0,1.0
"""2015_02_NE_BUF""",2015,"""REG""",2,"""2015-09-20""","""Sunday""","""13:00""","""NE""",40,"""BUF""",32,"""Home""",-8,72,0,"""2015092001""",56520,null,"""201509200buf""",3428,"""400791664""",null,10,7,112,-124,2.0,-103,-107,45.0,-105,-105,1,"""outdoors""","""a_turf""",62,7,"""00-0019596""","""00-0028118""","""Tom Brady""","""Tyrod Taylor""","""Bill Belichick""","""Rex Ryan""","""Ron Torbert""","""BUF00""","""Ralph Wilson Stadium""",0,1.0,27.0,14.0,1.0,28.0,21.0,0.0,-1.0,7.0,1.0,27.0,14.0,13.0,1.0,28.0,21.0,7.0,0.0,-1.0,7.0,6.0
"""2015_02_HOU_CAR""",2015,"""REG""",2,"""2015-09-20""","""Sunday""","""13:00""","""HOU""",17,"""CAR""",24,"""Home""",7,41,0,"""2015092000""",56521,null,"""201509200car""",3431,"""400791628""",null,7,7,129,-143,3.0,-117,106,41.0,-108,-102,0,"""outdoors""","""grass""",88,6,"""00-0028012""","""00-0027939""","""Ryan Mallett""","""Cam Newton""","""Bill O'Brien""","""Ron Rivera""","""Clete Blakeman""","""CAR00""","""Bank of America Stadium""",1,1.0,20.0,9.0,0.0,20.0,27.0,1.0,0.0,18.0,1.0,20.0,9.0,11.0,0.0,20.0,27.0,-7.0,1.0,0.0,18.0,18.0
"""2015_02_ARI_CHI""",2015,"""REG""",2,"""2015-09-20""","""Sunday""","""13:00""","""ARI""",48,"""CHI""",23,"""Home""",-25,71,0,"""2015092006""",56522,null,"""201509200chi""",3425,"""400791661""",null,7,7,-119,108,-2.0,-103,-107,45.5,-105,-105,0,"""outdoors""","""grass""",70,6,"""00-0021429""","""00-0024226""","""Carson Palmer""","""Jay Cutler""","""Bruce Arians""","""John Fox""","""Jerome Boger""","""CHI98""","""Soldier Field""",0,0.0,23.0,31.0,1.0,31.0,19.0,-1.0,-8.0,-12.0,0.0,23.0,31.0,-8.0,1.0,31.0,19.0,12.0,-1.0,-8.0,-12.0,-20.0
"""2015_02_SD_CIN""",2015,"""REG""",2,"""2015-09-20""","""Sunday""","""13:00""","""SD""",19,"""CIN""",24,"""Home""",5,43,0,"""2015092002""",56523,null,"""201509200cin""",3426,"""400791666""",null,7,7,156,-173,3.5,-110,100,47.5,-108,-102,0,"""outdoors""","""fieldturf""",69,6,"""00-0022942""","""00-0027973""","""Philip Rivers""","""Andy Dalton""","""Mike McCoy""","""Marvin Lewis""","""Jeff Triplette""","""CIN00""","""Paul Brown Stadium""",1,1.0,33.0,13.0,1.0,33.0,28.0,0.0,0.0,15.0,1.0,33.0,13.0,20.0,1.0,33.0,28.0,5.0,0.0,0.0,15.0,15.0


## 3. Define Features and Target

The model predicts whether the home team wins.

Features represent the difference between the two teams before the game:

- `win_pct_diff`: difference in previous win percentage
- `ppg_diff`: difference in points scored per game
- `defense_diff`: difference in points allowed per game

The target is:

- `home_win = 1`: home team won
- `home_win = 0`: home team lost

In [51]:
baseline_features = [
    "win_pct_diff",
    "ppg_diff",
    "defense_diff",
]

rolling_features = [
    "rolling_win_pct_diff_5",
    "rolling_ppg_diff_5",
    "rolling_defense_diff_5",
    "rolling_point_diff_diff_5",
]

features = baseline_features + rolling_features

target = "home_win"

In [52]:
print("Features:")
for feature in features:
    print("-", feature)

print("\nNumber of features:", len(features))

Features:
- win_pct_diff
- ppg_diff
- defense_diff
- rolling_win_pct_diff_5
- rolling_ppg_diff_5
- rolling_defense_diff_5
- rolling_point_diff_diff_5

Number of features: 7


## 4. Chronological Train / Validation / Test Split

To avoid data leakage, the model is evaluated chronologically.

- Training: 2015–2022
- Validation: 2023
- Test: 2024–2025

The validation set is used to compare models.

The test set is kept separate until the final evaluation.

In [53]:
train_data = model_data.filter(
    pl.col("season") <= 2022
)

validation_data = model_data.filter(
    pl.col("season") == 2023
)

test_data = model_data.filter(
    pl.col("season") >= 2024
)

## 5. Convert Data for Scikit-Learn

Polars is used for data preparation, while scikit-learn receives NumPy arrays for model training.

In [54]:
X_train = train_data.select(features).to_numpy()
y_train = train_data.select(target).to_numpy().ravel()

X_val = validation_data.select(features).to_numpy()
y_val = validation_data.select(target).to_numpy().ravel()

X_test = test_data.select(features).to_numpy()
y_test = test_data.select(target).to_numpy().ravel()

In [55]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (1950, 7)
y_train: (1950,)
X_val: (256, 7)
y_val: (256,)
X_test: (512, 7)
y_test: (512,)


In [56]:
print("Training rows:", X_train.shape[0])
print("Validation rows:", X_val.shape[0])
print("Test rows:", X_test.shape[0])

print("\nMissing values:")
print(model_data.select(features).null_count())

Training rows: 1950
Validation rows: 256
Test rows: 512

Missing values:
shape: (1, 7)
┌──────────────┬──────────┬──────────────┬──────────────┬──────────────┬─────────────┬─────────────┐
│ win_pct_diff ┆ ppg_diff ┆ defense_diff ┆ rolling_win_ ┆ rolling_ppg_ ┆ rolling_def ┆ rolling_poi │
│ ---          ┆ ---      ┆ ---          ┆ pct_diff_5   ┆ diff_5       ┆ ense_diff_5 ┆ nt_diff_dif │
│ u32          ┆ u32      ┆ u32          ┆ ---          ┆ ---          ┆ ---         ┆ f_5         │
│              ┆          ┆              ┆ u32          ┆ u32          ┆ u32         ┆ ---         │
│              ┆          ┆              ┆              ┆              ┆             ┆ u32         │
╞══════════════╪══════════╪══════════════╪══════════════╪══════════════╪═════════════╪═════════════╡
│ 0            ┆ 0        ┆ 0            ┆ 0            ┆ 0            ┆ 0           ┆ 0           │
└──────────────┴──────────┴──────────────┴──────────────┴──────────────┴─────────────┴─────────────┘


## 6. Logistic Regression

Logistic Regression provides a simple baseline model.

It estimates the probability that the home team wins based on the difference between the teams' pre-game statistics.

In [57]:
logistic_model = LogisticRegression(
    random_state=42
)

logistic_model.fit(
    X_train,
    y_train
)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following a

## 7. Random Forest

Random Forest can capture non-linear relationships between the team statistics.

The model uses multiple decision trees and averages their predictions.

In [58]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=10,
    random_state=42
)

rf_model.fit(
    X_train,
    y_train
)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap

## 8. Gradient Boosting

Gradient Boosting builds trees sequentially, with each tree attempting to improve the errors of the previous trees.

In [59]:
gb_model = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42
)

gb_model.fit(
    X_train,
    y_train
)

,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.05
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",200
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",15
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",1.0
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""Categorical"" and ""Enum"" are considered to be categorical features. The input must be a dataframe that is supported by narwhals (or supports it): :func:`narwhals.from_native` must work. This is the case, for instance, for pandas and polars DataFrames.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing va

## 9. Validation Predictions

We first evaluate the models on the 2023 validation season.

The models output probabilities rather than only predicting a winner.

In [60]:
logistic_val_prob = logistic_model.predict_proba(X_val)[:, 1]
rf_val_prob = rf_model.predict_proba(X_val)[:, 1]
gb_val_prob = gb_model.predict_proba(X_val)[:, 1]

## 10. Validation Metrics

Three metrics are used:

**Log Loss**

Measures the quality of predicted probabilities. Incorrect predictions with high confidence are penalized heavily.

**Brier Score**

Measures the mean squared difference between predicted probabilities and the actual outcome.

**Accuracy**

Measures how often the predicted winner is correct using a 50% probability threshold.

For probability prediction, Log Loss and Brier Score are more informative than accuracy alone.

In [61]:
results = pl.DataFrame({
    "model": [
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    
    "log_loss": [
        log_loss(y_val, logistic_val_prob),
        log_loss(y_val, rf_val_prob),
        log_loss(y_val, gb_val_prob)
    ],
    
    "brier_score": [
        brier_score_loss(y_val, logistic_val_prob),
        brier_score_loss(y_val, rf_val_prob),
        brier_score_loss(y_val, gb_val_prob)
    ],
    
    "accuracy": [
        accuracy_score(y_val, logistic_val_prob >= 0.5),
        accuracy_score(y_val, rf_val_prob >= 0.5),
        accuracy_score(y_val, gb_val_prob >= 0.5)
    ]
})

results

model,log_loss,brier_score,accuracy
str,f64,f64,f64
"""Logistic Regression""",0.670916,0.238852,0.59375
"""Random Forest""",0.685942,0.245741,0.5859375
"""Gradient Boosting""",0.717765,0.257085,0.546875


In [62]:
results.sort("log_loss")

model,log_loss,brier_score,accuracy
str,f64,f64,f64
"""Logistic Regression""",0.670916,0.238852,0.59375
"""Random Forest""",0.685942,0.245741,0.5859375
"""Gradient Boosting""",0.717765,0.257085,0.546875


## 11. Example Win Probabilities

The model produces a probability that the home team wins rather than simply predicting the winner.

This allows us to distinguish between confident and uncertain predictions.

In [63]:
predictions = test_data.select([
    "season",
    "week",
    "home_team",
    "away_team",
    "home_win",
]).with_columns([
    pl.Series(
        "logistic_home_win_prob",
        logistic_model.predict_proba(X_test)[:, 1]
    ),
    
    pl.Series(
        "rf_home_win_prob",
        rf_model.predict_proba(X_test)[:, 1]
    ),
    
    pl.Series(
        "gb_home_win_prob",
        gb_model.predict_proba(X_test)[:, 1]
    )
])

In [64]:
predictions.head(20)

season,week,home_team,away_team,home_win,logistic_home_win_prob,rf_home_win_prob,gb_home_win_prob
i32,i32,str,str,i8,f64,f64,f64
2024,2,"""MIA""","""BUF""",0,0.454426,0.546602,0.682147
2024,2,"""BAL""","""LV""",0,0.637747,0.62665,0.656032
2024,2,"""CAR""","""LAC""",0,0.18838,0.420928,0.665371
2024,2,"""DAL""","""NO""",0,0.389543,0.42037,0.438946
2024,2,"""DET""","""TB""",0,0.444611,0.519393,0.499349
…,…,…,…,…,…,…,…
2024,2,"""PHI""","""ATL""",0,0.845652,0.65502,0.654785
2024,3,"""NYJ""","""NE""",1,0.56325,0.585585,0.623144
2024,3,"""CLE""","""NYG""",0,0.690354,0.774314,0.849647


## 12. Final Test Evaluation

After comparing the models on the 2023 validation season, we evaluate them on the unseen 2024–2025 test seasons.

The test set represents games that were not used during model training or model comparison.

In [65]:
final_results = pl.DataFrame({
    "model": [
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    
    "log_loss": [
        log_loss(
            y_test,
            logistic_model.predict_proba(X_test)[:, 1]
        ),
        log_loss(
            y_test,
            rf_model.predict_proba(X_test)[:, 1]
        ),
        log_loss(
            y_test,
            gb_model.predict_proba(X_test)[:, 1]
        )
    ],
    
    "brier_score": [
        brier_score_loss(
            y_test,
            logistic_model.predict_proba(X_test)[:, 1]
        ),
        brier_score_loss(
            y_test,
            rf_model.predict_proba(X_test)[:, 1]
        ),
        brier_score_loss(
            y_test,
            gb_model.predict_proba(X_test)[:, 1]
        )
    ],
    
    "accuracy": [
        accuracy_score(
            y_test,
            logistic_model.predict_proba(X_test)[:, 1] >= 0.5
        ),
        accuracy_score(
            y_test,
            rf_model.predict_proba(X_test)[:, 1] >= 0.5
        ),
        accuracy_score(
            y_test,
            gb_model.predict_proba(X_test)[:, 1] >= 0.5
        )
    ]
})

final_results.sort("log_loss")

model,log_loss,brier_score,accuracy
str,f64,f64,f64
"""Random Forest""",0.636091,0.222829,0.6328125
"""Logistic Regression""",0.639771,0.224045,0.65625
"""Gradient Boosting""",0.66609,0.235537,0.619141


In [66]:
final_results.sort("log_loss")

model,log_loss,brier_score,accuracy
str,f64,f64,f64
"""Random Forest""",0.636091,0.222829,0.6328125
"""Logistic Regression""",0.639771,0.224045,0.65625
"""Gradient Boosting""",0.66609,0.235537,0.619141


## 13. Conclusion

Three machine learning approaches were evaluated for predicting NFL home-team wins:

- Logistic Regression
- Random Forest
- Gradient Boosting

The models were trained using pre-game team statistics and evaluated chronologically to reduce the risk of data leakage.

The primary evaluation metrics were Log Loss and Brier Score because the objective is to produce reliable win probabilities rather than only predict the winner.

The next stage of the project will improve the feature set by incorporating additional information such as:

- Rolling team performance
- Offensive and defensive EPA
- Turnover performance
- Quarterback performance
- Rest and scheduling
- Home-field effects
- Potentially betting-market information

The final goal is to produce calibrated game-level win probabilities and identify the factors contributing to each prediction.

### Baseline Results

The baseline models were evaluated on the unseen 2024–2025 test seasons.

Random Forest achieved the lowest Log Loss (0.640) and Brier Score (0.224), while Logistic Regression achieved the highest accuracy (65.63%). Gradient Boosting performed worse across all three metrics.

Because the main objective of the project is to predict win probabilities rather than only classify winners, Log Loss and Brier Score are particularly important.

The results provide a baseline against which more advanced feature engineering can be evaluated.

## V2 Results — Rolling Performance

The second version of the model added rolling five-game team statistics to the baseline season-level features.

Compared with the baseline, all three models achieved lower Log Loss and Brier Score on the 2024–2025 test set. This indicates that recent team performance provided additional information for estimating win probabilities.

The improvement was relatively small, suggesting that rolling performance alone does not dramatically improve the predictor.

Accuracy did not consistently improve, reinforcing the importance of evaluating probability quality separately from winner classification.

The next version will introduce efficiency-based features, including offensive and defensive performance metrics.